# Notebook 3: Sentiment Analysis at Scale

**Business problem:** Do audiences respond differently to sponsored vs organic content?
This notebook applies RoBERTa (cardiffnlp/twitter-roberta-base-sentiment-latest)
to score all 55,000+ comments as positive, neutral, or negative.

**Validation:** Compared against 200 manually labelled comments.
**Architecture decision:** RoBERTa over VADER — social-media tuned contextual model
vs keyword matching. Validated F1 justifies the compute cost.


In [ ]:
# !pip install transformers torch wordcloud scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from tqdm import tqdm
from sklearn.metrics import classification_report

DATA_DIR = '../data/'
merged_comments_df = pd.read_csv(DATA_DIR + 'merged_comments.csv')
print(merged_comments_df.shape)


In [ ]:
traditional_comments_df.shape
traditional_comments_df.info()
traditional_comments_df.describe()
traditional_comments_df

In [ ]:
#Drop rows with missing commentText
traditional_comments_df = traditional_comments_df.dropna(subset=['commentText'])
traditional_comments_df.loc[:, 'commentPublishedAt'] = pd.to_datetime(traditional_comments_df['commentPublishedAt'])

#Remove comments that are too short to be meaningful (e.g., ≤3 characters)
traditional_comments_df = traditional_comments_df[traditional_comments_df['commentText'].str.len() > 3]

#Standardize text (lowercase, strip whitespace)
traditional_comments_df.loc[:,'commentText'] = traditional_comments_df['commentText'].str.strip().copy()

print(f"After cleaning: {traditional_comments_df.shape}")
print(traditional_comments_df.head(3))

In [ ]:
# Only these columns needed from videos
video_info = traditional_videos_df[['videoId', 'Sponsorship', 'genre', 'sizeCategory']]

# Merge onto comments
merged_comments_df = traditional_comments_df.merge(video_info, on='videoId', how='left')

print(merged_comments_df[['videoId', 'Sponsorship', 'genre', 'sizeCategory']].head())
print(merged_comments_df[['videoId', 'Sponsorship', 'genre', 'sizeCategory']].drop_duplicates().head(10))

In [ ]:
# EDA analysis on comments section

# Count number of comments for each video
comment_counts = merged_comments_df.groupby('videoId').size().reset_index(name='num_comments')

# Merge back sponsorship, genre, size info
comment_counts = comment_counts.merge(
    traditional_videos_df[['videoId', 'Sponsorship', 'genre', 'sizeCategory']],
    on='videoId', how='left'
)

#Plot Comment Counts by Sponsorship
plt.figure(figsize=(9, 5))
sns.boxplot(data=comment_counts, x='Sponsorship', y='num_comments', showfliers=False)
plt.xlabel('Sponsorship Type')
plt.ylabel('Number of Comments per Video')
plt.title('Audience Comment Activity by Sponsorship Type')
plt.tight_layout()
plt.show()

#By Genre and Size
plt.figure(figsize=(9, 5))
sns.boxplot(data=comment_counts, x='genre', y='num_comments', hue='Sponsorship', showfliers=False)
plt.xlabel('Genre')
plt.ylabel('Number of Comments per Video')
plt.title('Comments per Video by Genre and Sponsorship')
plt.legend(title='Sponsorship')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.boxplot(data=comment_counts, x='sizeCategory', y='num_comments', hue='Sponsorship', showfliers=False)
plt.xlabel('Creator Size')
plt.ylabel('Number of Comments per Video')
plt.title('Comments per Video by Size and Sponsorship')
plt.legend(title='Sponsorship')
plt.tight_layout()
plt.show()


In [ ]:

# Comments per sponsorship type
sponsorship_counts = merged_comments_df['Sponsorship'].value_counts()
print(sponsorship_counts)

# Comments per genre and size
print(merged_comments_df.groupby(['genre', 'sizeCategory']).size().unstack())


In [ ]:
# Distribution of like counts
print(merged_comments_df['commentLikeCount'].describe())

# Distribution of comment text length
merged_comments_df['commentLength'] = merged_comments_df['commentText'].str.len()
print(merged_comments_df['commentLength'].describe())


In [ ]:
#Count Distribution of Comments: Comment likes are extremely skewed. A histogram (log scale) will show the distribution and help you see if most engagement is “broad” or driven by a few viral comments.

plt.figure(figsize=(8,4))
plt.hist(merged_comments_df['commentLikeCount'] + 1, bins=100, log=True)  # +1 to avoid log(0)
plt.xlabel('Comment Like Count (log scale)')
plt.ylabel('Number of Comments')
plt.title('Distribution of Comment Like Counts')
plt.xscale('log')
plt.tight_layout()
plt.show()


In [ ]:
#Comment Length by Sponsorship Type: Check if audience writes longer (or shorter) comments under sponsored, implicit, or organic videos. Longer comments can sometimes indicate stronger opinions.

plt.figure(figsize=(8,5))
sns.boxplot(data=merged_comments_df, x='Sponsorship', y='commentLength', showfliers=False)
plt.xlabel('Sponsorship Type')
plt.ylabel('Comment Length')
plt.title('Comment Length by Sponsorship Type')
plt.tight_layout()
plt.show()

In [ ]:
#Number of Comments per Video by Sponsorship Type: Are sponsored videos actually driving more discussion? This is a core research question.

comments_per_video = merged_comments_df.groupby(['videoId', 'Sponsorship']).size().reset_index(name='numComments')
sns.boxplot(data=comments_per_video, x='Sponsorship', y='numComments', showfliers=False)
plt.ylabel('Number of Comments per Video')
plt.title('Engagement: Comments per Video by Sponsorship')
plt.tight_layout()
plt.show()

In [ ]:
#Genre × Size × Sponsorship Breakdown : Let’s see where the comments are coming from. Are some genres/sizes overrepresented? This will help you know if your comparison groups are balanced.
pivot = pd.pivot_table(merged_comments_df, index='genre', columns=['sizeCategory', 'Sponsorship'], values='commentText', aggfunc='count', fill_value=0)
print(pivot)


In [ ]:
#Most Common Words (Optional, Fast Preview): Quick word frequency preview to see if audience talks differently under sponsored vs. organic videos. This is not topic modeling, just word counts.

from collections import Counter
import re

def get_word_counts(comments):
    words = []
    for text in comments:
        words += re.findall(r'\w+', text.lower())
    return Counter(words).most_common(20)

for sponsor_type in ['Explicit', 'Implicit', 'Organic']:
    print(f"\nMost common words in {sponsor_type} videos:")
    subset = merged_comments_df[merged_comments_df['Sponsorship'] == sponsor_type]['commentText']
    print(get_word_counts(subset))


In [ ]:
#Comment Like Count by Sponsorship Type: This checks whether sponsored/organic videos attract more “liked” (popular/viral) comments, adding nuance to your engagement analysis.

plt.figure(figsize=(8,5))
sns.boxplot(data=merged_comments_df, x='Sponsorship', y='commentLikeCount', showfliers=False)
plt.xlabel('Sponsorship Type')
plt.ylabel('Comment Like Count')
plt.title('Comment Like Count by Sponsorship Type')
plt.tight_layout()
plt.show()

In [ ]:
#Comment Activity Over Calendar Time: Shows if engagement spikes around sponsored uploads, or if organic content generates more sustained discussion.

# Comments per day, colored by sponsorship type
# Always make sure your datetime column is actually datetime!
merged_comments_df['commentPublishedAt'] = pd.to_datetime(merged_comments_df['commentPublishedAt'], errors='coerce')
merged_comments_df['commentDate'] = merged_comments_df['commentPublishedAt'].dt.date

activity_by_day = merged_comments_df.groupby(['commentDate', 'Sponsorship']).size().reset_index(name='numComments')

plt.figure(figsize=(12,5))
sns.lineplot(data=activity_by_day, x='commentDate', y='numComments', hue='Sponsorship')
plt.xlabel('Date')
plt.ylabel('Number of Comments')
plt.title('Comment Activity Over Time by Sponsorship')
plt.tight_layout()
plt.show()



In [ ]:
# Ensure all comments are linked to a known video with sponsorship, genre, size
missing = merged_comments_df[['Sponsorship', 'genre', 'sizeCategory']].isnull().sum()
print("Missing values by column:\n", missing)

Collectively, these findings confirm that the dataset is both rich and multi-faceted. Patterns of audience engagement differ across sponsorship type, genre, and influencer size—not only in volume, but in depth and intensity of participation. Therefore, the next phase of analysis will use advanced NLP models to quantify the sentiment of audience comments, enabling a deeper understanding of how sponsorship shapes audience attitudes and creator reputation.

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import pipeline

# Use the CardiffNLP Twitter-RoBERTa model for sentiment (best for YouTube)
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0,           # or -1 if you’re on CPU
    truncation=True,
    max_length=256
)

In [ ]:
def batch_sentiment(ids, texts, batch_size=64):
    rows = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_ids = ids[i : i + batch_size]
        batch_texts = texts[i : i + batch_size]
        preds = sentiment_model(batch_texts)
        for cid, p in zip(batch_ids, preds):
            rows.append({
                "commentId": cid,
                "sent_label_raw": p["label"],
                "sent_score":       p["score"]
            })
    return pd.DataFrame(rows)


# make sure you have a commentId in your comments DF
merged_comments_df["commentId"] = merged_comments_df.index.astype(str)

# run it
sent_df = batch_sentiment(
    merged_comments_df["commentId"].tolist(),
    merged_comments_df["commentText"].tolist()
)

# 3) map to human labels and merge back
label_map = {
    "LABEL_0": "negative",
    "LABEL_1": "neutral",
    "LABEL_2": "positive"
}
sent_df["sentiment_label"] = sent_df["sent_label_raw"].map(label_map)

merged_comments_df = merged_comments_df.merge(
    sent_df[["commentId","sentiment_label","sent_score"]],
    on="commentId",
    how="left"
)


In [ ]:
# 1) Drop any old columns if present
for c in ["sentiment_label","sent_score"]:
    if c in merged_comments_df:
        merged_comments_df = merged_comments_df.drop(columns=c)

# 2) Rename the raw model output and merge
sent_df = sent_df.rename(columns={"sent_label_raw":"sentiment_label"})

merged_comments_df = (
    merged_comments_df
    .merge(
        sent_df[["commentId","sentiment_label","sent_score"]],
        on="commentId",
        how="left"
    )
)

# 3) Quick sanity‐check
print( merged_comments_df[["sentiment_label","sent_score"]].head() )


In [ ]:
# or if you prefer to keep the first occurrence:
merged_comments_df = merged_comments_df.loc[:, ~merged_comments_df.columns.duplicated(keep="first")]

In [ ]:
merged_comments_df

In [ ]:
print(merged_comments_df.groupby('sentiment_label')['sent_score'].describe())

#Check Sentiment Score Distribution by Label
plt.figure(figsize=(8,5))
sns.histplot(data=merged_comments_df, x='sent_score', hue='sentiment_label', bins=30, kde=True, stat="density", common_norm=False)
plt.title('Sentiment Score Distribution by Predicted Label')
plt.xlabel('Model Sentiment Score (Confidence)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
manual_df = pd.read_csv('../data/Comments Validation.csv')
print(manual_df.columns)
manual_df['commentText'] = manual_df['commentText'].str.strip()
manual_df["commentId"] = manual_df.index.astype(str)
manual_df["LABELS"] = manual_df["LABELS"].str.lower()

print(manual_df.head())

In [ ]:
compare_df = manual_df.merge(
    merged_comments_df[["commentText","sentiment_label"]],
    on="commentText",
    how="inner"
)

labels = ["positive","neutral","negative"]
y_true = compare_df["LABELS"]
y_pred = compare_df["sentiment_label"]

print(f"Validation set size: {len(compare_df)}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred, labels=labels))
print("\nClassification report:")
print(classification_report(y_true, y_pred, labels=labels, digits=3))

In [ ]:
#Distribution of comments per video: Checks if a few videos dominate audience discussion or if comments are spread out, helps justify filtering or normalizing later
comment_counts = merged_comments_df['videoId'].value_counts()
plt.figure(figsize=(8,5))
plt.hist(comment_counts, bins=30, color='mediumslateblue', edgecolor='black')
plt.xlabel('Comments per Video')
plt.ylabel('Number of Videos')
plt.title('Distribution of Comment Counts per Video')
plt.tight_layout()
plt.show()

In [ ]:
#Comment like counts by sponsorship type: Shows if comments under sponsored (Explicit/Implicit) videos get more “likes.” Likes often mean more agreement or engagement. Can suggest whether sponsored content sparks more popular discussions (good or bad!).
plt.figure(figsize=(8,5))
sns.boxplot(x='Sponsorship', y='commentLikeCount', data=merged_comments_df, showfliers=False)
plt.yscale('log')
plt.xlabel('Sponsorship Type')
plt.ylabel('Comment Like Count (log scale)')
plt.title('Distribution of Comment Likes by Sponsorship')
plt.tight_layout()
plt.show()

In [ ]:
#Sentiment distribution across sponsorship types: Directly checks your main question: Are comments on sponsored videos more positive/negative/neutral? Use this to argue if sponsorship affects “audience mood.”
sent_dist = merged_comments_df.groupby('Sponsorship')['sentiment_label'].value_counts(normalize=True).unstack().fillna(0)
sent_dist.plot(kind='bar', stacked=True, figsize=(8,5), colormap='coolwarm')
plt.title('Comment Sentiment by Sponsorship')
plt.ylabel('Fraction of Comments')
plt.xlabel('Sponsorship Type')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

# By genre: Shows if audience mood is different for Tech vs Lifestyle, or Nano/Micro/Macro creators.Useful for clustering later! Justifies your grouping choices (“Genre matters for sentiment distribution…”).
genre_sent = merged_comments_df.groupby(['genre', 'sentiment_label']).size().unstack(fill_value=0)
genre_sent_norm = genre_sent.div(genre_sent.sum(axis=1), axis=0)
genre_sent_norm[['positive', 'neutral', 'negative']].plot(kind='bar', stacked=True, figsize=(10,5))
plt.title('Comment Sentiment Distribution by Genre')
plt.ylabel('Proportion')
plt.tight_layout()
plt.show()

# By sizeCategory
size_sent = merged_comments_df.groupby(['sizeCategory', 'sentiment_label']).size().unstack(fill_value=0)
size_sent_norm = size_sent.div(size_sent.sum(axis=1), axis=0)
size_sent_norm[['positive', 'neutral', 'negative']].plot(kind='bar', stacked=True, figsize=(8,5))
plt.title('Comment Sentiment Distribution by Creator Size')
plt.ylabel('Proportion')
plt.tight_layout()
plt.show()



In [ ]:
#Temporal Trends: Comments Over Time by Sponsorship: Detects if sponsored videos drive short-term spikes in engagement. See if there's a pattern (like more engagement around launches or campaigns). Helps with your time series/causal analysis setup!
merged_comments_df['commentDate'] = pd.to_datetime(merged_comments_df['commentPublishedAt']).dt.date
comments_per_day = merged_comments_df.groupby(['commentDate', 'Sponsorship']).size().unstack().fillna(0)
comments_per_day.rolling(7).mean().plot(figsize=(10,6))
plt.ylabel('7-Day Avg Comments per Day')
plt.title('Comment Volume Over Time by Sponsorship Type')
plt.xlabel('Date')
plt.show()


In [ ]:
#Pivot Table: Comments Count by Sponsorship, Genre, and Size: Spot imbalances or sparsity. If some groups have very few comments, your findings might be less robust there.
pivot = merged_comments_df.pivot_table(
    index='Sponsorship',
    columns=['genre', 'sizeCategory'],
    values='commentText',
    aggfunc='count',
    fill_value=0
)
print(pivot)


In [ ]:
#Sentiment Scores Distribution by Sponsorship/Genre/Size
plt.figure(figsize=(8,5))
sns.boxplot(x='Sponsorship', y='sent_score', data=merged_comments_df)
plt.title('Sentiment Score Distribution by Sponsorship Type')
plt.xlabel('Sponsorship')
plt.ylabel('Sentiment Score')
plt.tight_layout()
plt.show()

# Also try a countplot of labels:
plt.figure(figsize=(8,5))
sns.countplot(x='sentiment_label', hue='Sponsorship', data=merged_comments_df)
plt.title('Comment Sentiment Labels by Sponsorship')
plt.xlabel('Sentiment')
plt.ylabel('Number of Comments')
plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud

for sponsor_type in ['Explicit', 'Implicit', 'Organic']:
    text = ' '.join(merged_comments_df[merged_comments_df['Sponsorship'] == sponsor_type]['commentText'])
    wc = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10,5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Most Common Words: {sponsor_type}')
    plt.show()


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

for sentiment in ['positive', 'neutral', 'negative']:
    text = ' '.join(merged_comments_df[merged_comments_df['sentiment_label'] == sentiment]['commentText'].dropna())
    plt.figure(figsize=(8,6))
    if text.strip():
        wc = WordCloud(width=600, height=400, background_color='white').generate(text)
        plt.imshow(wc, interpolation='bilinear')
        plt.axis('off')
        plt.title(f"Word Cloud: {sentiment.capitalize()} Comments")
    else:
        plt.text(0.5, 0.5, "No Data", ha='center', va='center', fontsize=20)
        plt.axis('off')
        plt.title(f"Word Cloud: {sentiment.capitalize()} Comments")
    plt.tight_layout()
    plt.show()


In [ ]:
for genre in merged_comments_df['genre'].unique():
    text = ' '.join(merged_comments_df[merged_comments_df['genre'] == genre]['commentText'].dropna())
    plt.figure(figsize=(8,6))
    if text.strip():
        wc = WordCloud(width=600, height=400, background_color='white').generate(text)
        plt.imshow(wc, interpolation='bilinear')
        plt.axis('off')
        plt.title(f"Word Cloud: {genre} Comments")
    else:
        plt.text(0.5, 0.5, "No Data", ha='center', va='center', fontsize=20)
        plt.axis('off')
        plt.title(f"Word Cloud: {genre} Comments")
    plt.tight_layout()
    plt.show()


In [ ]:
merged_comments_df['commentPublishedAt'] = pd.to_datetime(merged_comments_df['commentPublishedAt'], errors='coerce')
merged_comments_df['commentDate'] = merged_comments_df['commentPublishedAt'].dt.date

sentiment_time = merged_comments_df.groupby(['commentDate', 'sentiment_label']).size().unstack(fill_value=0)
sentiment_time.plot(figsize=(14,6), marker='o')
plt.title('Sentiment Over Time (All Comments)')
plt.xlabel('Date')
plt.ylabel('Number of Comments')
plt.tight_layout()
plt.show()


In [ ]:
merged_comments_df.describe()
merged_comments_df.columns

In [ ]:
from scipy.stats import skew, kurtosis, entropy

# 1. Aggregate all desired metrics per video
agg_df = merged_comments_df.groupby('videoId').agg(
    mean_sentiment_score=('sent_score', 'mean'),
    median_sentiment_score=('sent_score', 'median'),
    std_sentiment_score=('sent_score', 'std'),
    num_comments=('commentText', 'count'),
    avg_comment_length=('commentLength', 'mean'),
    avg_comment_like=('commentLikeCount', 'mean'),
    max_comment_like=('commentLikeCount', 'max'),
    min_comment_like=('commentLikeCount', 'min'),
    num_positive=('sentiment_label', lambda x: (x == 'positive').sum()),
    num_neutral=('sentiment_label', lambda x: (x == 'neutral').sum()),
    num_negative=('sentiment_label', lambda x: (x == 'negative').sum()),
)


agg_df = agg_df.reset_index()

# 2. Proportions and dominant label
agg_df['pct_positive'] = agg_df['num_positive'] / agg_df['num_comments']
agg_df['pct_neutral'] = agg_df['num_neutral'] / agg_df['num_comments']
agg_df['pct_negative'] = agg_df['num_negative'] / agg_df['num_comments']
agg_df['dominant_sentiment'] = agg_df[['num_positive','num_neutral','num_negative']].idxmax(axis=1).str.replace('num_','')


# 3. Skew, kurtosis, mode, entropy, and outliers
# Calculate and reset index for all, then merge with agg_df for robustness

skew_series = merged_comments_df.groupby('videoId')['sent_score'].apply(lambda x: skew(x, nan_policy='omit')).reset_index()
kurtosis_series = merged_comments_df.groupby('videoId')['sent_score'].apply(lambda x: kurtosis(x, nan_policy='omit')).reset_index()
mode_sentiment = merged_comments_df.groupby('videoId')['sentiment_label'].agg(lambda x: x.mode().iat[0]).reset_index()
sentiment_diversity = merged_comments_df.groupby('videoId')['sentiment_label'].apply(lambda x: entropy(x.value_counts(normalize=True))).reset_index(name='sentiment_diversity')
outlier_high = merged_comments_df.groupby('videoId')['sent_score'].max().reset_index()
outlier_low = merged_comments_df.groupby('videoId')['sent_score'].min().reset_index()
time_range = merged_comments_df.groupby('videoId')['commentPublishedAt'].agg(lambda x: (x.max() - x.min()).days).reset_index(name='comment_time_range_days')

# Merge these back to agg_df
agg_df = agg_df.merge(skew_series.rename(columns={'sent_score':'sentiment_skew'}), on='videoId', how='left')
agg_df = agg_df.merge(kurtosis_series.rename(columns={'sent_score':'sentiment_kurtosis'}), on='videoId', how='left')
agg_df = agg_df.merge(mode_sentiment.rename(columns={'sentiment_label':'mode_sentiment'}), on='videoId', how='left')
agg_df = agg_df.merge(sentiment_diversity, on='videoId', how='left')
agg_df = agg_df.merge(time_range, on='videoId', how='left')
agg_df = agg_df.merge(outlier_high.rename(columns={'sent_score':'max_sentiment_score'}), on='videoId', how='left')
agg_df = agg_df.merge(outlier_low.rename(columns={'sent_score':'min_sentiment_score'}), on='videoId', how='left')

agg_df['has_extreme_positive'] = agg_df['max_sentiment_score'] > 0.98   # tune as needed
agg_df['has_extreme_negative'] = agg_df['min_sentiment_score'] < 0.4     # tune as needed

# 4. Join aggregated sentiment/engagement metrics
master_video_df = df.merge(
    agg_df,  # This is your video-level aggregation
    on='videoId',
    how='left'
)

# 5. Inspect results
print('Aggregated Columns:', agg_df.columns)
print('After Merging Comments at Video Level:', master_video_df.columns)
print(master_video_df.shape)